# 01 · Real headlines (EPFL format): fallback when the data is not arcade

Use this only if the runner can't read the data, i.e. real dated headlines instead of `ins_*/oos_*` sessions. The README describes the plan and the embedding bake-off.

Steps: point `df` and `holdout` at the real tables → set the column names → run all. With `DEMO = "dated"` it runs on a synthetic dated table, so the notebook is always ready.

In [ ]:
%load_ext autoreload
%autoreload 2
import math
from pathlib import Path

import numpy as np
import pandas as pd

from arena.config import TaskSpec
from arena.cv import cross_validate, walk_forward_by_time
from arena.metrics import score_by_unit
from arena.models import make_tfidf_ridge
from arena.paths import SUBMISSIONS, default_data_dir

In [ ]:
DEMO = "dated"  # "dated" = synthetic rehearsal table; None = the real files below
TIME, TEXT, TARGET, ID, TICKER = "time", "text", "target", "row_id", "ticker"  # CHANGE to the real column names
if DEMO == "dated":
    from arena.synth import make_dated_headlines
    full = make_dated_headlines().reset_index(drop=True).rename_axis(ID).reset_index()
    cut = full[TIME].quantile(0.8)
    df, holdout = full[full[TIME] <= cut].copy(), full[full[TIME] > cut].drop(columns=TARGET)
else:
    DATA_DIR = default_data_dir()
    df = pd.read_parquet(DATA_DIR / "CHANGE_ME_train.parquet")      # rows with the target
    holdout = pd.read_parquet(DATA_DIR / "CHANGE_ME_holdout.parquet")  # rows to predict, no target
df = df.sort_values(TIME).reset_index(drop=True)
print(len(df), "training rows |", len(holdout), "holdout rows |", df[TIME].min(), "->", df[TIME].max())
df.head()

## Walk-forward CV by date (train on the past only, 1-day gap)

In [ ]:
folds = walk_forward_by_time(df[TIME], k=5, min_train_frac=0.5, gap=pd.Timedelta(days=1))
y_txt = df[TARGET].to_numpy()

def tfidf_fit_predict(alpha=10.0, min_df=3, ngram=(1, 2)):
    def fp(tr, te):
        pipe = make_tfidf_ridge(alpha=alpha, min_df=min_df, ngram_range=ngram)
        pipe.fit(df.loc[tr, TEXT], y_txt[tr])
        return pipe.predict(df.loc[te, TEXT])
    return fp

print(cross_validate(tfidf_fit_predict(), y_txt, folds))

## Robust tuning: maximise mean − std of fold Sharpes (not the best single backtest)

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    res = cross_validate(tfidf_fit_predict(alpha=trial.suggest_float("alpha", 0.3, 300, log=True),
                                           min_df=trial.suggest_int("min_df", 1, 10)), y_txt, folds)
    return res.mean - res.std

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))  # same pick every run
study.optimize(objective, n_trials=8, timeout=120)
study.best_params, study.best_value

## If HRT scores per day: score the out-of-fold predictions summed per day

In [ ]:
from arena.cv import out_of_fold

best = study.best_params
# optimistic: alpha and min_df were picked on these same folds; compare ideas at fixed settings instead
oof = out_of_fold(tfidf_fit_predict(**best), len(df), folds, allow_partial=True)
ok = ~np.isnan(oof)
print("per row:", round(cross_validate(tfidf_fit_predict(**best), y_txt, folds).mean, 3),
      "| per day:", round(score_by_unit(oof[ok], y_txt[ok], df.loc[ok, TIME].dt.date), 3))

## Upload: fit on all training rows, predict the holdout, write the CSV

In [ ]:
PRED = "pred"  # CHANGE to the upload's column names
VOL = None     # CHANGE to a past-volatility column of `holdout`, if there is one (README, real-headline fallback)
spec = TaskSpec(name="real-headlines", halfway_bar=0, exit_bar=1, annualization=math.sqrt(252), ddof=1,
                demean_returns=False, id_col=ID, pred_col=PRED)
pipe = make_tfidf_ridge(alpha=best["alpha"], min_df=best["min_df"]).fit(df[TEXT], y_txt)
pred = pipe.predict(holdout[TEXT])
from arena.sizing import clip_sd, vol_scale
from arena.submission import write_submission

if VOL is not None:
    pred = vol_scale(pred, holdout[VOL])  # size by past volatility
pred = clip_sd(pred, 3.0)  # cap at 3 s.d.
path = write_submission(holdout[ID].to_numpy(), pred, SUBMISSIONS / "real_01_tfidf.csv", spec)
print(path)

## Optional: headline embeddings (README) and FinBERT, on the Apple GPU
Needs `uv sync --extra nlp` and the models cached beforehand. Set `HF_HUB_OFFLINE=1` on the day.

In [ ]:
try:
    import torch
    from transformers import AutoModel, AutoTokenizer, pipeline
except ImportError:
    torch = None
    print("transformers not installed: uv sync --extra nlp")

DEVICE = "mps" if torch is not None and torch.backends.mps.is_available() else "cpu"

def embed(texts, name="Alibaba-NLP/gte-modernbert-base", batch=128):
    """CLS embeddings of unique texts (max 64 tokens), batched on the GPU."""
    tok, model = AutoTokenizer.from_pretrained(name), AutoModel.from_pretrained(name).to(DEVICE).eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch):
            enc = tok(texts[i:i + batch], padding=True, truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
            out.append(model(**enc).last_hidden_state[:, 0].float().cpu().numpy())
    return np.concatenate(out)

if torch is not None:
    clf = pipeline("text-classification", model="ProsusAI/finbert", top_k=None, truncation=True,
                   device=DEVICE, batch_size=64)
    print(clf(df[TEXT].head(3).str.slice(0, 400).tolist()))

## Main text model (§9a): gte-modernbert embeddings + Ridge, on the same walk-forward folds
Each unique headline is embedded once and cached in `data/` (gitignored). On real data the signal (walk-forward OOF + holdout
predictions) is exported for a blend with TF-IDF; the vectors are not (too big for git).

In [ ]:
GTE = "Alibaba-NLP/gte-modernbert-base"  # fast fallback: "BAAI/bge-small-en-v1.5" (about 1 min for 137k headlines)
GTE_ALPHAS = (10.0, 100.0, 1000.0, 10000.0)  # short grid, declared in advance
if torch is not None:
    import hashlib
    import os

    from sklearn.linear_model import Ridge

    from arena.paths import DATA, SIGNALS
    from arena.signals import export_signal

    texts = pd.Index(pd.concat([df[TEXT], holdout[TEXT]]).astype(str).unique())
    key = hashlib.sha1("\n".join(texts).encode()).hexdigest()[:10]
    cache = DATA / f"emb_{GTE.split('/')[-1]}_{key}.npy"
    vecs = np.load(cache) if cache.exists() else embed(texts.tolist(), name=GTE)
    np.save(cache, vecs)
    x_train = vecs[texts.get_indexer(df[TEXT].astype(str))]
    x_hold = vecs[texts.get_indexer(holdout[TEXT].astype(str))]

    def gte_fit_predict(alpha):
        def fp(tr, te):
            return Ridge(alpha=alpha).fit(x_train[tr], y_txt[tr]).predict(x_train[te])
        return fp

    gte_cv = {a: cross_validate(gte_fit_predict(a), y_txt, folds) for a in GTE_ALPHAS}
    for a, res in gte_cv.items():
        print(f"gte + Ridge alpha={a:g}: {res}")
    GTE_ALPHA = max(gte_cv, key=lambda a: gte_cv[a].mean - gte_cv[a].std)  # robust pick, like the TF-IDF tuning
    gte_hold = Ridge(alpha=GTE_ALPHA).fit(x_train, y_txt).predict(x_hold)
    if DEMO is None:  # real data only: never write demo signals into signals/
        ME = os.environ.get("ARENA_ME", "me")  # CHANGE to your folder name
        path = export_signal(SIGNALS, ME, "gte_ridge", gte_fit_predict(GTE_ALPHA), df[ID].to_numpy(),
                             holdout[ID].to_numpy(), gte_hold, folds=folds)
        print("exported", path, "-> blend it with the TF-IDF signal")